In [17]:
import pandas as pd
import sqlite3
import os

os.chdir('/Users/prathamesh018/Desktop/GLP1 white area/glp1-prescriber-whitespace-analysis')
print(f"Working directory: {os.getcwd()}")

df = pd.read_csv("data/processed/glp1_clean.csv")

conn = sqlite3.connect("data/processed/glp1.db")
df.to_sql("prescribers", conn, if_exists="replace", index=False)

print("Database ready")
print(f"Loaded {len(df)} rows into prescribers table")

Working directory: /Users/prathamesh018/Desktop/GLP1 white area/glp1-prescriber-whitespace-analysis
Database ready
Loaded 220 rows into prescribers table


In [18]:
query = """
SELECT 
    first_name,
    last_name,
    city,
    state,
    specialty,
    SUM(total_claims) as total_claims,
    SUM(total_patients) as total_patients,
    ROUND(SUM(total_cost), 2) as total_cost
FROM prescribers
GROUP BY npi, first_name, last_name, city, state, specialty
ORDER BY total_claims DESC
LIMIT 20
"""

top_prescribers = pd.read_sql_query(query, conn)

print(top_prescribers)

    first_name   last_name            city state            specialty  \
0          Sam  Fereidouni      Scottsdale    AZ      Family Practice   
1       Herman    Blomeier          Skokie    IL        Endocrinology   
2      Janelle      Hinson        Columbia    SC  Physician Assistant   
3      Natalie   Philbrick  Corpus Christi    TX    Internal Medicine   
4   Bhavandeep       Bajaj       Baltimore    MD    Internal Medicine   
5        Robin       House          London    KY   Nurse Practitioner   
6      Lakshmi  Srinivasan         Fremont    CA        Endocrinology   
7       Anoopa       Koshy         Chicago    IL        Endocrinology   
8     Jennifer     Burgart        Asheboro    NC      Family Practice   
9        Treah    Haggerty      Morgantown    WV      Family Practice   
10        Jeri      Savoie       Amsterdam    NY   Nurse Practitioner   
11        Troy      Mohler    Purcellville    VA      Family Practice   
12       Viral       Patel      Crittenden    KY   

In [19]:
query = """
SELECT
    first_name,
    last_name,
    city,
    state,
    specialty,
    SUM(total_claims) as total_claims,
    CASE
        WHEN SUM(total_claims) >= 500 THEN 'Tier 1 - High Value'
        WHEN SUM(total_claims) >= 100 THEN 'Tier 2 - Medium Value'
        ELSE 'Tier 3 - Low Value'
    END as target_tier
FROM prescribers
GROUP BY npi, first_name, last_name, city, state, specialty
ORDER BY total_claims DESC
"""

tiers = pd.read_sql_query(query, conn)

print(tiers['target_tier'].value_counts())
print()
print(tiers.head(10))

target_tier
Tier 2 - Medium Value    13
Tier 1 - High Value       6
Tier 3 - Low Value        3
Name: count, dtype: int64

   first_name   last_name            city state            specialty  \
0         Sam  Fereidouni      Scottsdale    AZ      Family Practice   
1      Herman    Blomeier          Skokie    IL        Endocrinology   
2     Janelle      Hinson        Columbia    SC  Physician Assistant   
3     Natalie   Philbrick  Corpus Christi    TX    Internal Medicine   
4  Bhavandeep       Bajaj       Baltimore    MD    Internal Medicine   
5       Robin       House          London    KY   Nurse Practitioner   
6     Lakshmi  Srinivasan         Fremont    CA        Endocrinology   
7      Anoopa       Koshy         Chicago    IL        Endocrinology   
8    Jennifer     Burgart        Asheboro    NC      Family Practice   
9       Treah    Haggerty      Morgantown    WV      Family Practice   

   total_claims            target_tier  
0          2505    Tier 1 - High Value  
1 

In [20]:
tiers.to_csv("data/processed/glp1_hcp_tiers.csv", index=False)

top_prescribers.to_csv("data/processed/glp1_top_prescribers.csv", index=False)

print("Files saved")

Files saved
